In [1]:
import os
import pandas as pd
import numpy as np
from numba import njit, float64, int64, uint64,types
from numba.typed import Dict

In [2]:

# 현재 파일들이 있는 그 위치 그대로 설정
Base_dir = "C:/Users/user/Desktop/IDS_masters/9) Car-Hacking Dataset"

# 파일 이름에 포함된 단어로 공격 유형 구분
attack_mapping = {
    "Dos": 1,
    "Fuzzing": 2,
    "Spoofing":4
}

attack_files = []

# 폴더 안을 바로 검사
for attack_name, attack_id in attack_mapping.items():
    attack_dir = os.path.join(Base_dir, attack_name)
    if not os.path.isdir(attack_dir):
        continue

    for fname in os.listdir(attack_dir):
        if fname.endswith(".csv"):
            attack_files.append({
                "path": os.path.join(attack_dir, fname),
                "attack_id": attack_id
            })

In [3]:
#################################
# 2. Visualization Mirgu Dataset
#################################
hash_cache = {}

# [ADD] payload 8바이트 리스트로 만드는 함수 (너가 쓰던 스타일)
def parse_payload(row):
    # row에는 b0~b7 컬럼이 있고, 이미 0패딩되어 있음
    return [int(row[f"b{i}"]) for i in range(8)]

def process_csv_file(path, attack_id):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            parts = line.split(",")
            if len(parts) < 4:
                continue

            ts_str, canid_raw, dlc_str = parts[0], parts[1], parts[2]
            label = parts[-1].strip()         # [MINOR] strip
            data_tokens = parts[3:-1]

            # dlc/ts 파싱
            try:
                ts = float(ts_str)
                dlc = int(dlc_str)
            except:
                continue

            # payload bytes: DLC 만큼만 읽고, 8바이트로 0 패딩
            payload = []
            for i in range(min(dlc, len(data_tokens), 8)):
                tok = data_tokens[i].strip()
                if tok == "" or tok.lower() == "nan":
                    payload.append(0)
                else:
                    try:
                        payload.append(int(tok, 16))
                    except:
                        payload.append(0)

            payload += [0] * (8 - len(payload))
            payload = payload[:8]

            rows.append([ts, canid_raw, dlc, *payload, label])

    df = pd.DataFrame(
        rows,
        columns=["timestamp", "CAN_ID", "DLC"] + [f"b{i}" for i in range(8)] + ["Label"]
    )

    # CAN_ID int 변환
    df["int_CAN_ID"] = df["CAN_ID"].apply(lambda x: int(str(x).strip(), 16)).astype(np.int64)


    # Payloads 컬럼 추가 
    df["Payloads"] = df.apply(parse_payload, axis=1).tolist()

    # 라벨링
    df["Labeling"] = df["Label"].map({"T": attack_id, "R": 0}).fillna(0).astype(int)

    df = df[["timestamp","int_CAN_ID","DLC","Payloads","Labeling"]]

    return df


In [4]:
from numba import njit, float64, int64, uint64, types
from numba.typed import Dict
import numpy as np

@njit
def popcount64(x):
    c = 0
    v = int64(x)
    while v:
        v &= v - int64(1)
        c += 1
    return c

@njit
def pack_payload_u64_dlc(row, dlc):
    v = uint64(0)
    d = dlc
    if d < 0:
        d = 0
    if d > 8:
        d = 8
    for i in range(d):
        v |= uint64(row[i]) << (i * 8)
    return v

@njit(fastmath=True)
def calculate_features(timestamps, can_ids, dlcs, payloads):
    n = len(timestamps)

    # 12개 피처 공간 확보 (0~11)
    features = np.zeros((n, 12), dtype=np.float64)

    last_time_map    = Dict.empty(key_type=types.int64, value_type=types.float64)
    last_payload_map = Dict.empty(key_type=types.int64, value_type=types.uint64)

    local_cnt_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    id_ham_ema    = Dict.empty(key_type=types.int64, value_type=types.float64)
    streak_map    = Dict.empty(key_type=types.int64, value_type=types.int64)

    # IAT용 EMA
    iat_ema_map    = Dict.empty(key_type=types.int64, value_type=types.float64)
    iat_sq_ema_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    alpha_iat = 0.001

    # EMA baseline for freq
    ema_freq_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    alpha_freq = 0.01  # (실제 업데이트는 alpha_freq_gate 사용)

    # index-gap CV용 (현재는 사용 안 함)
    last_pos_map = Dict.empty(key_type=types.int64, value_type=types.int64)
    gap_n_map    = Dict.empty(key_type=types.int64, value_type=types.int64)
    gap_mean_map = Dict.empty(key_type=types.int64, value_type=types.float64)
    gap_M2_map   = Dict.empty(key_type=types.int64, value_type=types.float64)

    alpha_ham = 0.05
    eps = 1e-9
    W = 256.0
    WARMUP = 1000  # 초기 구간: baseline만 학습

    top1_id = int64(-1)
    top2_id = int64(-1)
    top1_cnt = 0.0
    top2_cnt = 0.0

    sum_pos_log_ratio = 0.0

    for i in range(n):
        # window reset
        if (i % 256) == 0:
            local_cnt_map.clear()
            last_pos_map.clear()
            gap_n_map.clear()
            gap_mean_map.clear()
            gap_M2_map.clear()

            top1_id = int64(-1)
            top2_id = int64(-1)
            top1_cnt = 0.0
            top2_cnt = 0.0
            sum_pos_log_ratio = 0.0

        cid = int64(can_ids[i])
        dlc = int64(dlcs[i])
        row = payloads[i]

        # ---- IAT ----
        curr_iat = 0.0
        if cid in last_time_map:
            curr_iat = timestamps[i] - last_time_map[cid]
            if curr_iat < 0.0:
                curr_iat = 0.0
        last_time_map[cid] = timestamps[i]

        iat_rel = 1.0
        if cid in iat_ema_map:
            mean_iat = iat_ema_map[cid]
            iat_rel = curr_iat / (mean_iat + 1e-3)
            # 너무 극단적으로 빠를 때는 baseline 업데이트를 막아 오염 방지
            if iat_rel > 0.5:
                iat_ema_map[cid] = (1.0 - alpha_iat) * mean_iat + alpha_iat * curr_iat
        else:
            iat_ema_map[cid] = curr_iat
            iat_sq_ema_map[cid] = curr_iat * curr_iat
            iat_rel = 1.0

        # ---- DLC clamp ----
        if dlc < 0:
            dlc = 0
        elif dlc > 8:
            dlc = 8
        cur_bytes = pack_payload_u64_dlc(row, dlc)

        # ---- Payload Hamming & Streak ----
        h_dist = 0.0
        rel_change = 0.0
        streak = int64(0)
        if cid in last_payload_map:
            diff_bits = cur_bytes ^ last_payload_map[cid]
            h_dist = float64(popcount64(diff_bits))
            if diff_bits == uint64(0):
                streak = streak_map.get(cid, int64(0)) + int64(1)
            else:
                streak = int64(0)
            streak_map[cid] = streak

            if cid in id_ham_ema:
                avg_h = id_ham_ema[cid]
                rel_change = h_dist / (avg_h + 0.1)
                id_ham_ema[cid] = (1.0 - alpha_ham) * avg_h + alpha_ham * h_dist
            else:
                rel_change = 1.0
                id_ham_ema[cid] = h_dist
        else:
            streak_map[cid] = int64(0)
        last_payload_map[cid] = cur_bytes

        # ---- Entropy ----
        ent = 0.0
        if dlc > 0:
            p_counts = np.zeros(256, dtype=np.int64)
            for k in range(dlc):
                p_counts[row[k]] += 1
            for c in p_counts:
                if c > 0:
                    p = c / float64(dlc)
                    ent -= p * np.log(p)

        # ---- ID Entropy ----
        id_ent = 0.0
        if i >= 255:
            win_id_counts = Dict.empty(key_type=types.int64, value_type=types.float64)
            for j in range(i - 255, i + 1):
                wid = int64(can_ids[j])
                win_id_counts[wid] = win_id_counts.get(wid, 0.0) + 1.0
            for k_id in win_id_counts:
                pk = win_id_counts[k_id] / W
                id_ent -= pk * np.log(pk + eps)
            id_ent = id_ent / 4.85

        # ---- Freq Local ----
        old_cnt = local_cnt_map.get(cid, 0.0)
        cnt = old_cnt + 1.0
        local_cnt_map[cid] = cnt
        freq_local = cnt / W

        # ---- Freq Log Ratio ----
        alpha_freq_gate = 0.001
        freeze_thr = 0.3
        log_ratio = 0.0
        if cid in ema_freq_map:
            ema = ema_freq_map[cid]
            log_ratio = np.log((freq_local + eps) / (ema + eps))
            if log_ratio > 0.0:
                sum_pos_log_ratio += log_ratio
            # 너무 튀는 구간은 baseline 업데이트 중지
            if log_ratio < freeze_thr:
                ema_freq_map[cid] = (1.0 - alpha_freq_gate) * ema + alpha_freq_gate * freq_local
        else:
            ema_freq_map[cid] = freq_local

        # ---- Top1/Top2 Dominance ----
        if cid == top1_id:
            top1_cnt = cnt
        elif cid == top2_id:
            top2_cnt = cnt

        if cnt > top1_cnt + 1e-12:
            if cid != top1_id:
                top2_id, top2_cnt = top1_id, top1_cnt
            top1_id, top1_cnt = cid, cnt
        elif cnt > top2_cnt + 1e-12 and cid != top1_id:
            top2_id, top2_cnt = cid, cnt

        # ---- 공통 피처 저장 ----
        features[i, 0]  = 1.0 if cid == 0 else 0.0
        features[i, 1]  = float64(dlc) / 8.0
        features[i, 2]  = np.log1p(rel_change) / 5.0
        features[i, 3]  = np.log1p(ent * rel_change)
        features[i, 4]  = freq_local
        features[i, 5]  = id_ent
        features[i, 6]  = float64(streak) / W
        features[i, 7]  = np.log((cnt + eps) / (top1_cnt + eps))
        features[i, 8]  = top1_cnt / W
        features[i, 9]  = (top1_cnt - top2_cnt) / (top1_cnt + eps)

        # ---- 워밍업 / 본격 탐지 분기 ----
        if i < WARMUP:
            # 초기 구간: baseline만 학습, 간격/윈도우 이상치 피처는 0으로
            features[i, 10] = 0.0
            features[i, 11] = 0.0
        else:
            # 이후 구간: 평소 대비 IAT / 윈도우 freq 이상치 사용
            features[i, 10] = np.log1p(1.0 / (iat_rel + 1e-3))
            features[i, 11] = sum_pos_log_ratio / W  # 윈도우 전체 빈도 이상치 (정규화)

    return features

In [5]:
# ==========================================
# 4. Making Feature with Numba
# ==========================================

def Make_feature(path, attack_id):

    df = process_csv_file(path, attack_id)

    # ======== to numpy ========== #
    timestamps = df["timestamp"].to_numpy(np.float32)
    can_ids = df["int_CAN_ID"].to_numpy(np.int64)
    payloads = np.array(df["Payloads"].tolist(), dtype=np.uint8)
    dlcs = df["DLC"].to_numpy(np.int64)
    labels = df["Labeling"].to_numpy(np.int64)

    
    # ======== calculate feature ========== #
    feature9 = calculate_features(timestamps, can_ids,dlcs ,payloads)
    print(feature9.shape)

    return feature9, labels

In [6]:
# ==========================================
# 5. Slide Window and Label
# ==========================================
def Sliding_Window_and_Labeling(feature, label, win_size=256, stride=128):
    windows = []
    labels = []
    n = feature.shape[0]
    for start in range(0, n-win_size+1 , stride):
        end = start + win_size
        windows.append(feature[start:end])
        labels.append(label[start:end])


    return (
        np.stack(windows, axis=0).astype(np.float32),
        np.stack(labels, axis=0).astype(np.int64)
    )

In [7]:
# ==========================================
# 6. main
# ==========================================
all_x = []
all_y = []

for item in attack_files:
    feature9, labels = Make_feature(item["path"], item["attack_id"]) # 각 feature 추출
    windows, y = Sliding_Window_and_Labeling(feature9,labels) # 윈도우 만들기

    all_x.append(windows)
    all_y.append(y)

all_x_win = np.concatenate(all_x, axis=0)
all_y_win = np.concatenate(all_y, axis=0)

(3665771, 12)
(3838860, 12)
(4443142, 12)
(4621702, 12)


In [8]:
# ==========================================
# 7. Save
# ==========================================
np.savez(
    "C:/Users/user/Desktop/IDS_masters/dataset/carhacking_test_0223_135.npz",
    X = all_x_win.astype(np.float32),
    y = all_y_win.astype(np.int64)
    )

print(f" Saved dataset")

print("X shape:", all_x_win.shape)
print("y shape:", all_y_win.shape)

 Saved dataset
X shape: (129444, 256, 12)
y shape: (129444, 256)
